In [1]:
import os
import pandas as pd

# 1. Definimos las rutas del proyecto
RUTA_RAW = os.path.join("data", "raw")
RUTA_INTERIM = os.path.join("data", "interim")

ruta_raw_kn = os.path.join(RUTA_RAW, "all_kindle_review .csv")
ruta_salida_kn = os.path.join(RUTA_INTERIM, "interacciones_kindle.csv")

# 2. Leemos únicamente las 4 columnas necesarias para no saturar memoria
cols_kn = ['reviewerID', 'asin', 'rating', 'reviewTime']
df_kn = pd.read_csv(ruta_raw_kn, usecols=cols_kn)

# 3. Renombramos al esquema canónico
df_kn = df_kn.rename(columns={
    'reviewerID': 'user_id',
    'asin': 'product_id',
    'rating': 'rating',
    'reviewTime': 'fecha'
})

# 4. Agregamos el prefijo 'kn_' para aislar el dominio
df_kn['product_id'] = 'kn_' + df_kn['product_id'].astype(str)

# 5. Estandarizamos la fecha a formato AAAA-MM-DD
df_kn['fecha'] = pd.to_datetime(df_kn['fecha'], errors='coerce').dt.strftime('%Y-%m-%d')

# 6. Limpieza: eliminamos nulos y duplicados de usuario-producto
df_kn = df_kn.dropna(subset=['user_id', 'product_id', 'rating'])
df_kn = df_kn.drop_duplicates(subset=['user_id', 'product_id'])

# 7. Guardamos el archivo físico independiente
df_kn.to_csv(ruta_salida_kn, index=False)

print(f"Kindle procesado con éxito: {len(df_kn):,} filas.")
print(f"Guardado en: {ruta_salida_kn}")
display(df_kn.head(3))

Kindle procesado con éxito: 12,000 filas.
Guardado en: data\interim\interacciones_kindle.csv


,product_id,rating,fecha,user_id
0,kn_B0033UV8HI,3,2010-09-02,A3HHXRELK8BHQG
1,kn_B002HJV4DE,5,2013-10-08,A2RGNZ0TRF578I
2,kn_B002ZG96I4,3,2014-04-11,A3S0H2HV6U1I7F


In [2]:
import os
import json
import pandas as pd

# 1. Definimos rutas
ruta_raw_vg = os.path.join(RUTA_RAW, "Video_Games_5.json")
ruta_salida_vg = os.path.join(RUTA_INTERIM, "interacciones_videogames.csv")

# 2. Leemos línea a línea extrayendo solo las 4 variables canónicas
registros_vg = []
with open(ruta_raw_vg, "r", encoding="utf-8") as f:
    for linea in f:
        reg = json.loads(linea)
        registros_vg.append({
            'user_id': reg.get('reviewerID'),
            'product_id': 'vg_' + str(reg.get('asin')),
            'rating': float(reg.get('overall', 0)),
            'fecha': reg.get('unixReviewTime')
        })

# 3. Convertimos la lista a DataFrame de Pandas
df_vg = pd.DataFrame(registros_vg)

# 4. Formateamos la fecha (segundos a AAAA-MM-DD)
df_vg['fecha'] = pd.to_datetime(df_vg['fecha'], unit='s', errors='coerce').dt.strftime('%Y-%m-%d')

# 5. Limpieza: eliminamos filas con nulos y deduplicamos usuario-producto
df_vg = df_vg.dropna(subset=['user_id', 'product_id', 'rating'])
df_vg = df_vg.drop_duplicates(subset=['user_id', 'product_id'])

# 6. Guardamos físicamente en data/interim/
df_vg.to_csv(ruta_salida_vg, index=False)

print(f"Video Games procesado: {len(df_vg):,} filas.")
print(f"Guardado en: {ruta_salida_vg}")
display(df_vg.head(3))

Video Games procesado: 473,427 filas.
Guardado en: data\interim\interacciones_videogames.csv


,user_id,product_id,rating,fecha
0,A1HP7NVNPFMA4N,vg_0700026657,5.0,2015-10-17
1,A1JGAP0185YJI6,vg_0700026657,4.0,2015-07-27
2,A1YJWEXHQBWK2B,vg_0700026657,3.0,2015-02-23


In [3]:
import os
import pandas as pd

# 1. Definir rutas
ruta_raw_bt = os.path.join(RUTA_RAW, "ratings_Beauty.csv")
ruta_salida_bt = os.path.join(RUTA_INTERIM, "interacciones_beauty.csv")

# 2. Cargar únicamente las 4 columnas canónicas
cols_bt = ['UserId', 'ProductId', 'Rating', 'Timestamp']
df_bt = pd.read_csv(ruta_raw_bt, usecols=cols_bt)

# 3. Renombrar al estándar
df_bt = df_bt.rename(columns={
    'UserId': 'user_id',
    'ProductId': 'product_id',
    'Rating': 'rating',
    'Timestamp': 'fecha'
})

# 4. Inyectar prefijo de dominio 'bt_'
df_bt['product_id'] = 'bt_' + df_bt['product_id'].astype(str)

# 5. Estandarizar fecha (viene en segundos)
df_bt['fecha'] = pd.to_datetime(df_bt['fecha'], unit='s', errors='coerce').dt.strftime('%Y-%m-%d')

# 6. Limpieza básica de nulos y deduplicación
df_bt = df_bt.dropna(subset=['user_id', 'product_id', 'rating'])
df_bt = df_bt.drop_duplicates(subset=['user_id', 'product_id'])

# 7. Guardar en data/interim/
df_bt.to_csv(ruta_salida_bt, index=False)

print(f"Beauty procesado con éxito: {len(df_bt):,} filas.")
print(f"Guardado en: {ruta_salida_bt}")
display(df_bt.head(3))

Beauty procesado con éxito: 2,023,070 filas.
Guardado en: data\interim\interacciones_beauty.csv


,user_id,product_id,rating,fecha
0,A39HTATAQ9V7YF,bt_0205616461,5.0,2013-05-28
1,A3JM6GV9MNOF9X,bt_0558925278,3.0,2012-12-14
2,A1Z513UWSAAO0F,bt_0558925278,5.0,2014-07-07


In [4]:
import os
import pandas as pd

# 1. Definimos rutas
ruta_raw_bk = os.path.join(RUTA_RAW, "Books_rating.csv")
ruta_salida_bk = os.path.join(RUTA_INTERIM, "interacciones_books.csv")

# 2. Leemos únicamente las 4 columnas canónicas
cols_bk = ['User_id', 'Id', 'review/score', 'review/time']
df_bk = pd.read_csv(ruta_raw_bk, usecols=cols_bk)

# 3. Renombramos al estándar canónico
df_bk = df_bk.rename(columns={
    'User_id': 'user_id',
    'Id': 'product_id',
    'review/score': 'rating',
    'review/time': 'fecha'
})

# 4. Agregamos el prefijo 'bk_'
df_bk['product_id'] = 'bk_' + df_bk['product_id'].astype(str)

# 5. Estandarizamos la fecha (segundos a AAAA-MM-DD)
df_bk['fecha'] = pd.to_datetime(df_bk['fecha'], unit='s', errors='coerce').dt.strftime('%Y-%m-%d')

# 6. Limpieza: eliminamos nulos, fechas erróneas (< 1990) y duplicados
df_bk = df_bk.dropna(subset=['user_id', 'product_id', 'rating', 'fecha'])
df_bk = df_bk[df_bk['fecha'] >= '1990-01-01']
df_bk = df_bk.drop_duplicates(subset=['user_id', 'product_id'])

# 7. Guardamos en data/interim/
df_bk.to_csv(ruta_salida_bk, index=False)

print(f"Books procesado con éxito: {len(df_bk):,} filas.")
print(f"Guardado en: {ruta_salida_bk}")
display(df_bk.head(3))

Books procesado con éxito: 2,397,596 filas.
Guardado en: data\interim\interacciones_books.csv


,product_id,user_id,rating,fecha
0,bk_1882931173,AVCGYZL8FQQTD,4.0,1999-10-23
1,bk_0826414346,A30TK6U7DNS82R,5.0,2004-09-21
2,bk_0826414346,A3UH4UZ4RSVO82,5.0,2004-03-09


In [5]:
import os
import pandas as pd

# 1. Definimos las rutas
ruta_raw_af = os.path.join(RUTA_RAW, "amazon-fashion-800k+-user-reviews-dataset.csv")
ruta_salida_af = os.path.join(RUTA_INTERIM, "interacciones_fashion.csv")

# 2. Leemos únicamente las 4 columnas canónicas necesarias
cols_af = ['user_id', 'parent_asin', 'rating', 'timestamp']
df_af = pd.read_csv(ruta_raw_af, usecols=cols_af)

# 3. Renombramos al estándar canónico
df_af = df_af.rename(columns={
    'parent_asin': 'product_id',
    'timestamp': 'fecha'
})

# 4. Agregamos el prefijo 'af_'
df_af['product_id'] = 'af_' + df_af['product_id'].astype(str)

# 5. Convertimos la fecha (en milisegundos) a formato AAAA-MM-DD
df_af['fecha'] = pd.to_datetime(df_af['fecha'], unit='ms', errors='coerce').dt.strftime('%Y-%m-%d')

# 6. Limpieza: eliminamos nulos y duplicados de usuario-producto
df_af = df_af.dropna(subset=['user_id', 'product_id', 'rating', 'fecha'])
df_af = df_af.drop_duplicates(subset=['user_id', 'product_id'])

# 7. Guardamos en data/interim/
df_af.to_csv(ruta_salida_af, index=False)

print(f"Fashion 800k procesado con éxito: {len(df_af):,} filas.")
print(f"Guardado en: {ruta_salida_af}")
display(df_af.head(3))

Fashion 800k procesado con éxito: 861,924 filas.
Guardado en: data\interim\interacciones_fashion.csv


,rating,product_id,user_id,fecha
0,1.0,af_B07QFTMTLP,AHASEZ65RESN57BMGRV6QBM5DTIA,2019-08-06
1,1.0,af_B0764KKDN1,AE3AMA3QSOHFKV46JJAHTHMMIR6A,2021-05-30
2,1.0,af_B07J1WHVCP,AH4CFWQE2HTC5BSWIEF3LVLUFK6A,2019-08-08


In [6]:
import os
import json
import pandas as pd

# 1. Rutas
ruta_raw_el = os.path.join(RUTA_RAW, "Electronics_5.json")
ruta_salida_el = os.path.join(RUTA_INTERIM, "interacciones_electronics.csv")

# 2. Leemos el archivo JSON línea por línea para cuidar la RAM
registros_el = []
with open(ruta_raw_el, "r", encoding="utf-8") as f:
    for linea in f:
        reg = json.loads(linea)
        registros_el.append({
            'user_id': reg.get('reviewerID'),
            'product_id': 'el_' + str(reg.get('asin')),
            'rating': float(reg.get('overall', 0)),
            'fecha': reg.get('unixReviewTime')
        })

# 3. Convertimos a DataFrame de Pandas
df_el = pd.DataFrame(registros_el)

# 4. Formateamos fecha, eliminamos nulos y duplicados
df_el['fecha'] = pd.to_datetime(df_el['fecha'], unit='s', errors='coerce').dt.strftime('%Y-%m-%d')
df_el = df_el.dropna(subset=['user_id', 'product_id', 'rating', 'fecha'])
df_el = df_el.drop_duplicates(subset=['user_id', 'product_id'])

# 5. Guardamos en data/interim/
df_el.to_csv(ruta_salida_el, index=False)

print(f"Electronics procesado con éxito: {len(df_el):,} filas.")
print(f"Guardado en: {ruta_salida_el}")
display(df_el.head(3))

Electronics procesado con éxito: 1,689,188 filas.
Guardado en: data\interim\interacciones_electronics.csv


,user_id,product_id,rating,fecha
0,AO94DHGC771SJ,el_0528881469,5.0,2013-06-02
1,AMO214LNFCEI4,el_0528881469,1.0,2010-11-25
2,A3N7T0DY83Y4IG,el_0528881469,3.0,2010-09-09


In [7]:
import os
import pandas as pd

RUTA_INTERIM = os.path.join("data", "interim")
RUTA_PROCESSED = os.path.join("data", "processed")
os.makedirs(RUTA_PROCESSED, exist_ok=True)

# 1. Lista de los 6 archivos independientes generados en staging
archivos_interim = [
    "interacciones_kindle.csv",
    "interacciones_videogames.csv",
    "interacciones_beauty.csv",
    "interacciones_books.csv",
    "interacciones_fashion.csv",
    "interacciones_electronics.csv"
]

# 2. Leemos y concatenamos secuencialmente
dfs = []
for archivo in archivos_interim:
    ruta = os.path.join(RUTA_INTERIM, archivo)
    if os.path.exists(ruta):
        print(f"Cargando {archivo}...")
        df_temp = pd.read_csv(ruta)
        dfs.append(df_temp)
    else:
        print(f"Aviso: {archivo} no encontrado en interim.")

# Unimos todo en un solo DataFrame
df_total = pd.concat(dfs, ignore_index=True)
print(f"\nTotal filas combinadas antes de k-core: {len(df_total):,}")

# 3. Aplicación del filtro k-core iterativo (usuarios >= 2, productos >= 3)
min_user_ratings = 2
min_item_ratings = 3

filas_anteriores = len(df_total)
iteracion = 1

while True:
    # Contar interacciones por usuario y producto
    user_counts = df_total['user_id'].value_counts()
    item_counts = df_total['product_id'].value_counts()

    # Identificar IDs válidos
    valid_users = user_counts[user_counts >= min_user_ratings].index
    valid_items = item_counts[item_counts >= min_item_ratings].index

    # Filtrar usando isin
    df_total = df_total[
        df_total['user_id'].isin(valid_users) & 
        df_total['product_id'].isin(valid_items)
    ]
    
    filas_actuales = len(df_total)
    print(f"Iteración {iteracion}: quedan {filas_actuales:,} filas.")
    
    # Detener cuando no se eliminen más registros
    if filas_actuales == filas_anteriores:
        break
    filas_anteriores = filas_actuales
    iteracion += 1

# 4. Guardamos el archivo final consolidado
ruta_final_interacciones = os.path.join(RUTA_PROCESSED, "interacciones.csv")
df_total.to_csv(ruta_final_interacciones, index=False)

print(f"\n¡Consolidado exitoso!")
print(f"Archivo guardado en: {ruta_final_interacciones}")
print(f"Total usuarios únicos: {df_total['user_id'].nunique():,}")
print(f"Total productos únicos: {df_total['product_id'].nunique():,}")
display(df_total.head(3))

Cargando interacciones_kindle.csv...
Cargando interacciones_videogames.csv...
Cargando interacciones_beauty.csv...
Cargando interacciones_books.csv...
Cargando interacciones_fashion.csv...
Cargando interacciones_electronics.csv...

Total filas combinadas antes de k-core: 7,457,205
Iteración 1: quedan 4,968,038 filas.
Iteración 2: quedan 4,797,386 filas.
Iteración 3: quedan 4,757,451 filas.
Iteración 4: quedan 4,747,905 filas.
Iteración 5: quedan 4,744,777 filas.
Iteración 6: quedan 4,743,573 filas.
Iteración 7: quedan 4,743,052 filas.
Iteración 8: quedan 4,742,817 filas.
Iteración 9: quedan 4,742,681 filas.
Iteración 10: quedan 4,742,582 filas.
Iteración 11: quedan 4,742,507 filas.
Iteración 12: quedan 4,742,457 filas.
Iteración 13: quedan 4,742,426 filas.
Iteración 14: quedan 4,742,396 filas.
Iteración 15: quedan 4,742,369 filas.
Iteración 16: quedan 4,742,348 filas.
Iteración 17: quedan 4,742,334 filas.
Iteración 18: quedan 4,742,326 filas.
Iteración 19: quedan 4,742,321 filas.
Itera

,product_id,rating,fecha,user_id
0,kn_B0033UV8HI,3.0,2010-09-02,A3HHXRELK8BHQG
1,kn_B002HJV4DE,5.0,2013-10-08,A2RGNZ0TRF578I
2,kn_B002ZG96I4,3.0,2014-04-11,A3S0H2HV6U1I7F


In [8]:
import os
import json
import pandas as pd

RUTA_RAW = os.path.join("data", "raw")
RUTA_PROCESSED = os.path.join("data", "processed")

# Cargamos solo los productos que sobrevivieron en interacciones.csv
print("Cargando productos válidos desde interacciones.csv...")
df_inter = pd.read_csv(os.path.join(RUTA_PROCESSED, "interacciones.csv"), usecols=['product_id', 'rating'])

# Calculamos estadísticas base (rating promedio y total de reseñas)
stats_prod = df_inter.groupby('product_id').agg(
    rating_promedio=('rating', 'mean'),
    numero_resenas=('rating', 'count')
).reset_index()

stats_prod['rating_promedio'] = stats_prod['rating_promedio'].round(2)
productos_validos = set(stats_prod['product_id'])

# Diccionario global en memoria para guardar la metadata de todas las celdas
catalogo_meta = {}

print(f"Total productos únicos a catalogar: {len(productos_validos):,}")
display(stats_prod.head(3))

Cargando productos válidos desde interacciones.csv...
Total productos únicos a catalogar: 220,178


,product_id,rating_promedio,numero_resenas
0,af_B000VIUJLG,2.00,3
1,af_B002QSTHEE,1.75,4
2,af_B007CS43EW,2.50,6


In [3]:
import os
import pandas as pd

print("Construyendo mapeo de Books (Id -> Title -> Metadata)...")

ruta_books_ratings = os.path.join(RUTA_RAW, "Books_rating.csv")
ruta_books_meta = os.path.join(RUTA_RAW, "books_data.csv")

# 1. Creamos el diccionario que asocia Id con Title leyendo solo esas 2 columnas
cols_puente = ['Id', 'Title']
# Leemos en fragmentos de 200k filas para no agotar la RAM
id_a_titulo = {}

print("Paso 1: Leyendo correspondencia Id -> Title desde Books_rating.csv...")
for chunk in pd.read_csv(ruta_books_ratings, usecols=cols_puente, chunksize=200000):
    chunk = chunk.dropna(subset=['Id', 'Title'])
    for _, f in chunk.iterrows():
        p_id = "bk_" + str(f['Id'])
        if p_id in productos_validos and p_id not in id_a_titulo:
            id_a_titulo[p_id] = str(f['Title'])

print(f"IDs de libros encontrados en transacciones: {len(id_a_titulo):,}")

# 2. Leemos la metadata descriptiva desde books_data.csv
print("Paso 2: Leyendo descripción y categorías desde books_data.csv...")
df_meta = pd.read_csv(ruta_books_meta, usecols=['Title', 'description', 'categories'])
df_meta = df_meta.dropna(subset=['Title']).drop_duplicates(subset=['Title'])

dict_desc = dict(zip(df_meta['Title'], df_meta['description']))
dict_cat = dict(zip(df_meta['Title'], df_meta['categories']))

# 3. Guardamos los datos reales en nuestro diccionario global catalogo_meta
conteo_bk = 0
for p_id, titulo_libro in id_a_titulo.items():
    cat = dict_cat.get(titulo_libro, "Books")
    desc = dict_desc.get(titulo_libro, "")
    
    catalogo_meta[p_id] = {
        'titulo': titulo_libro,
        'categoria': str(cat) if pd.notna(cat) else 'Books',
        'descripcion': str(desc) if pd.notna(desc) else ''
    }
    conteo_bk += 1

print(f"\n¡Éxito! Books procesado: {conteo_bk:,} libros identificados con su título real.")
del df_meta, id_a_titulo  # Liberamos memoria RAM

Construyendo mapeo de Books (Id -> Title -> Metadata)...


NameError: name 'RUTA_RAW' is not defined

In [ ]:
import os
import pandas as pd

print("Extrayendo metadata de Fashion 800k (parent_asin -> title)...")
ruta_af = os.path.join(RUTA_RAW, "amazon-fashion-800k+-user-reviews-dataset.csv")

conteo_af = 0

if os.path.exists(ruta_af):
    # Leemos por fragmentos de 100k filas cargando solo parent_asin y title
    for chunk in pd.read_csv(ruta_af, usecols=['parent_asin', 'title'], chunksize=100000):
        chunk = chunk.dropna(subset=['parent_asin', 'title'])
        for _, fila in chunk.iterrows():
            p_id = "af_" + str(fila['parent_asin'])
            if p_id in productos_validos and p_id not in catalogo_meta:
                titulo_prenda = str(fila['title']).strip()
                catalogo_meta[p_id] = {
                    'titulo': titulo_prenda,
                    'categoria': 'Clothing, Shoes & Jewelry',
                    'descripcion': titulo_prenda
                }
                conteo_af += 1

print(f"\n¡Éxito! Fashion procesado: {conteo_af:,} prendas identificadas con su nombre real.")

In [1]:
print(f"Total productos en memoria ahora mismo: {len(catalogo_meta):,}")
# Mostramos un libro al azar de los que guardó:
primer_id = next(iter(catalogo_meta))
print(f"Ejemplo: {primer_id} -> {catalogo_meta[primer_id]}")

NameError: name 'catalogo_meta' is not defined

In [4]:
import os
import json
import pandas as pd

RUTA_RAW = os.path.join("data", "raw")
RUTA_PROCESSED = os.path.join("data", "processed")

print("Cargando productos válidos desde interacciones.csv...")
df_inter = pd.read_csv(os.path.join(RUTA_PROCESSED, "interacciones.csv"), usecols=['product_id', 'rating'])

stats_prod = df_inter.groupby('product_id').agg(
    rating_promedio=('rating', 'mean'),
    numero_resenas=('rating', 'count')
).reset_index()

stats_prod['rating_promedio'] = stats_prod['rating_promedio'].round(2)
productos_validos = set(stats_prod['product_id'])
catalogo_meta = {}

print(f"Memoria restaurada. Total productos únicos a catalogar: {len(productos_validos):,}")

Cargando productos válidos desde interacciones.csv...
Memoria restaurada. Total productos únicos a catalogar: 220,178


In [5]:
import os
import pandas as pd

print("Construyendo mapeo de Books (Id -> Title -> Metadata)...")

ruta_books_ratings = os.path.join(RUTA_RAW, "Books_rating.csv")
ruta_books_meta = os.path.join(RUTA_RAW, "books_data.csv")

cols_puente = ['Id', 'Title']
id_a_titulo = {}

print("Paso 1: Leyendo correspondencia Id -> Title desde Books_rating.csv...")
for chunk in pd.read_csv(ruta_books_ratings, usecols=cols_puente, chunksize=200000):
    chunk = chunk.dropna(subset=['Id', 'Title'])
    for _, f in chunk.iterrows():
        p_id = "bk_" + str(f['Id'])
        if p_id in productos_validos and p_id not in id_a_titulo:
            id_a_titulo[p_id] = str(f['Title'])

print(f"IDs de libros encontrados en transacciones: {len(id_a_titulo):,}")

print("Paso 2: Leyendo descripción y categorías desde books_data.csv...")
df_meta = pd.read_csv(ruta_books_meta, usecols=['Title', 'description', 'categories'])
df_meta = df_meta.dropna(subset=['Title']).drop_duplicates(subset=['Title'])

dict_desc = dict(zip(df_meta['Title'], df_meta['description']))
dict_cat = dict(zip(df_meta['Title'], df_meta['categories']))

conteo_bk = 0
for p_id, titulo_libro in id_a_titulo.items():
    cat = dict_cat.get(titulo_libro, "Books")
    desc = dict_desc.get(titulo_libro, "")
    
    catalogo_meta[p_id] = {
        'titulo': titulo_libro,
        'categoria': str(cat) if pd.notna(cat) else 'Books',
        'descripcion': str(desc) if pd.notna(desc) else ''
    }
    conteo_bk += 1

print(f"\n¡Éxito! Books procesado: {conteo_bk:,} libros identificados con su título real.")
del df_meta, id_a_titulo

Construyendo mapeo de Books (Id -> Title -> Metadata)...
Paso 1: Leyendo correspondencia Id -> Title desde Books_rating.csv...
IDs de libros encontrados en transacciones: 70,839
Paso 2: Leyendo descripción y categorías desde books_data.csv...

¡Éxito! Books procesado: 70,839 libros identificados con su título real.


In [6]:
import os
import pandas as pd

# 1. Definición segura de rutas
RUTA_RAW = os.path.join("data", "raw")
ruta_af = os.path.join(RUTA_RAW, "amazon-fashion-800k+-user-reviews-dataset.csv")

print("Extrayendo nombres de prendas desde Fashion 800k...")
conteo_af = 0

if os.path.exists(ruta_af):
    # 2. Lectura por fragmentos para proteger la memoria RAM
    for chunk in pd.read_csv(ruta_af, usecols=['parent_asin', 'title'], chunksize=100000):
        chunk = chunk.dropna(subset=['parent_asin', 'title'])
        for _, fila in chunk.iterrows():
            p_id = "af_" + str(fila['parent_asin'])
            
            # 3. Guardar solo si es un producto válido y no lo hemos registrado
            if p_id in productos_validos and p_id not in catalogo_meta:
                titulo_prenda = str(fila['title']).strip()
                catalogo_meta[p_id] = {
                    'titulo': titulo_prenda,
                    'categoria': 'Clothing, Shoes & Jewelry',
                    'descripcion': titulo_prenda
                }
                conteo_af += 1

print(f"\n¡Éxito! Fashion procesado: {conteo_af:,} prendas identificadas con su nombre real.")
print(f"Total productos acumulados en memoria: {len(catalogo_meta):,}")

Extrayendo nombres de prendas desde Fashion 800k...

¡Éxito! Fashion procesado: 444 prendas identificadas con su nombre real.
Total productos acumulados en memoria: 71,283


In [7]:
import os
import pandas as pd

# 1. Definición segura de rutas
RUTA_RAW = os.path.join("data", "raw")
ruta_bt_rec = os.path.join(RUTA_RAW, "Amazon_Beauty_Recommendation.csv")

print("Extrayendo metadata de Beauty desde Amazon_Beauty_Recommendation.csv...")
conteo_bt = 0

if os.path.exists(ruta_bt_rec):
    # 2. Leemos únicamente las 3 columnas descriptivas
    df_bt = pd.read_csv(ruta_bt_rec, usecols=['ProductId', 'ProductType', 'URL'])
    df_bt = df_bt.dropna(subset=['ProductId']).drop_duplicates(subset=['ProductId'])
    
    # 3. Iteramos para enriquecer los productos válidos en memoria
    for _, fila in df_bt.iterrows():
        p_id = "bt_" + str(fila['ProductId'])
        
        if p_id in productos_validos and p_id not in catalogo_meta:
            url_str = str(fila['URL'])
            # Limpiamos el texto de la URL para obtener un título legible
            if '/' in url_str and len(url_str.split('/')) > 3:
                nombre_producto = url_str.split('/')[3].replace('-', ' ').strip().title()
            else:
                nombre_producto = str(fila['ProductType']).strip().title()
                
            categoria_str = f"Beauty & Personal Care - {str(fila['ProductType']).strip().title()}"
            
            catalogo_meta[p_id] = {
                'titulo': nombre_producto,
                'categoria': categoria_str,
                'descripcion': f"Producto cosmético categoría {fila['ProductType']}"
            }
            conteo_bt += 1
            
    del df_bt  # Liberamos la tabla de la memoria RAM

print(f"\n¡Éxito! Beauty procesado: {conteo_bt:,} productos identificados con datos reales.")
print(f"Total productos acumulados en memoria: {len(catalogo_meta):,}")

Extrayendo metadata de Beauty desde Amazon_Beauty_Recommendation.csv...

¡Éxito! Beauty procesado: 23,762 productos identificados con datos reales.
Total productos acumulados en memoria: 95,045


In [8]:
import os
import json

# 1. Definición segura de rutas
RUTA_RAW = os.path.join("data", "raw")
ruta_vg = os.path.join(RUTA_RAW, "Video_Games_5.json")

print("Extrayendo metadata de Video Games desde Video_Games_5.json...")
conteo_vg = 0

if os.path.exists(ruta_vg):
    # 2. Lectura en streaming línea por línea para proteger la RAM
    with open(ruta_vg, "r", encoding="utf-8") as f:
        for linea in f:
            reg = json.loads(linea)
            asin = reg.get('asin')
            
            if asin:
                p_id = "vg_" + str(asin)
                
                # 3. Guardar solo si sobrevivió al k-core y no lo hemos registrado
                if p_id in productos_validos and p_id not in catalogo_meta:
                    summary_texto = str(reg.get('summary', '')).strip()
                    
                    catalogo_meta[p_id] = {
                        'titulo': f"Video Game ({asin})",
                        'categoria': 'Video Games',
                        'descripcion': summary_texto if summary_texto else 'Video Game product'
                    }
                    conteo_vg += 1

print(f"\n¡Éxito! Video Games procesado: {conteo_vg:,} juegos catalogados.")
print(f"Total productos acumulados en memoria: {len(catalogo_meta):,}")

Extrayendo metadata de Video Games desde Video_Games_5.json...

¡Éxito! Video Games procesado: 17,336 juegos catalogados.
Total productos acumulados en memoria: 112,381


In [9]:
import os
import json

# 1. Rutas seguras definidas localmente
RUTA_RAW = os.path.join("data", "raw")
ruta_el = os.path.join(RUTA_RAW, "Electronics_5.json")

print("Extrayendo metadata de Electronics desde Electronics_5.json...")
conteo_el = 0

if os.path.exists(ruta_el):
    # 2. Lectura streaming línea por línea para proteger la RAM
    with open(ruta_el, "r", encoding="utf-8") as f:
        for linea in f:
            reg = json.loads(linea)
            asin = reg.get('asin')
            
            if asin:
                p_id = "el_" + str(asin)
                
                # 3. Guardar solo si es un producto válido y no lo hemos anotado
                if p_id in productos_validos and p_id not in catalogo_meta:
                    summary_texto = str(reg.get('summary', '')).strip()
                    
                    catalogo_meta[p_id] = {
                        'titulo': f"Electronics Device ({asin})",
                        'categoria': 'Electronics',
                        'descripcion': summary_texto if summary_texto else 'Consumer Electronics product'
                    }
                    conteo_el += 1

print(f"\n¡Éxito! Electronics procesado: {conteo_el:,} productos catalogados.")
print(f"Total productos acumulados en memoria: {len(catalogo_meta):,}")

Extrayendo metadata de Electronics desde Electronics_5.json...

¡Éxito! Electronics procesado: 63,001 productos catalogados.
Total productos acumulados en memoria: 175,382


In [10]:
import os
import pandas as pd

RUTA_PROCESSED = os.path.join("data", "processed")

print("Ensamblando catálogo maestro de productos...")

# Mapeo de categorías por defecto según el prefijo de 3 caracteres
prefijos_categoria = {
    'kn_': 'Kindle Store',
    'vg_': 'Video Games',
    'bt_': 'Beauty & Personal Care',
    'bk_': 'Books',
    'af_': 'Clothing, Shoes & Jewelry',
    'el_': 'Electronics'
}

registros_finales = []

# Iteramos sobre la tabla base de estadísticas calculadas en la Celda 1
for _, row in stats_prod.iterrows():
    p_id = row['product_id']
    meta = catalogo_meta.get(p_id, {})
    
    cat_default = prefijos_categoria.get(p_id[:3], 'General')
    
    registros_finales.append({
        'product_id': p_id,
        'titulo': meta.get('titulo', f"Producto {p_id}"),
        'categoria': meta.get('categoria', cat_default),
        'rating_promedio': row['rating_promedio'],
        'numero_resenas': row['numero_resenas'],
        'descripcion': meta.get('descripcion', '')
    })

# Convertimos a DataFrame
df_productos = pd.DataFrame(registros_finales)

# Guardamos físicamente en disco el CSV procesado
ruta_final_prod = os.path.join(RUTA_PROCESSED, "productos.csv")
df_productos.to_csv(ruta_final_prod, index=False)

print("\n" + "="*50)
print(f"¡Catálogo maestro completado y guardado en disco!")
print(f"Ruta: {ruta_final_prod}")
print(f"Total de productos en el catálogo: {len(df_productos):,}")
print("="*50)

# Mostramos una muestra de 5 filas para verificar las columnas
display(df_productos.head(5))

Ensamblando catálogo maestro de productos...

¡Catálogo maestro completado y guardado en disco!
Ruta: data\processed\productos.csv
Total de productos en el catálogo: 220,178


,product_id,titulo,categoria,rating_promedio,numero_resenas,descripcion
0,af_B000VIUJLG,DON'T BUY THIS PRODUCT!,"Clothing, Shoes & Jewelry",2.00,3,DON'T BUY THIS PRODUCT!
1,af_B002QSTHEE,Waste of money,"Clothing, Shoes & Jewelry",1.75,4,Waste of money
2,af_B007CS43EW,One Star,"Clothing, Shoes & Jewelry",2.50,6,One Star
3,af_B007F55UGW,Don't buy.....save your money,"Clothing, Shoes & Jewelry",2.80,5,Don't buy.....save your money
4,af_B007TX6Z32,... not tell any difference in night vision a ...,"Clothing, Shoes & Jewelry",2.33,3,... not tell any difference in night vision a ...


In [11]:
import os
import pandas as pd

RUTA_PROCESSED = os.path.join("data", "processed")

print("Iniciando auditoría de calidad de datos...")
print("=" * 60)

# 1. Cargar archivos procesados
df_inter = pd.read_csv(os.path.join(RUTA_PROCESSED, "interacciones.csv"))
df_prod = pd.read_csv(os.path.join(RUTA_PROCESSED, "productos.csv"))

# --- TEST 1: Unicidad y claves primarias ---
duplicados_prod = df_prod['product_id'].duplicated().sum()
nulos_prod_id = df_prod['product_id'].isna().sum()

print(f"1. Unicidad de productos:")
print(f"   - IDs duplicados en catálogo: {duplicados_prod} {'[OK]' if duplicados_prod == 0 else '[ERROR]'}")
print(f"   - IDs nulos en catálogo: {nulos_prod_id} {'[OK]' if nulos_prod_id == 0 else '[ERROR]'}")

# --- TEST 2: Integridad Referencial Cruzada ---
items_inter = set(df_inter['product_id'])
items_prod = set(df_prod['product_id'])

huerfanos = items_inter - items_prod
print(f"\n2. Integridad Referencial:")
print(f"   - Productos en interacciones: {len(items_inter):,}")
print(f"   - Productos en catálogo: {len(items_prod):,}")
print(f"   - Ítems en interacciones sin ficha en catálogo: {len(huerfanos)} {'[OK]' if len(huerfanos) == 0 else '[ERROR]'}")

# --- TEST 3: Validación del k-core ---
conteo_usuarios = df_inter.groupby('user_id').size()
conteo_items = df_inter.groupby('product_id').size()

min_user = conteo_usuarios.min()
min_item = conteo_items.min()

print(f"\n3. Validación de k-core:")
print(f"   - Mínimo de reseñas por usuario: {min_user} (requerido >= 2) {'[OK]' if min_user >= 2 else '[ERROR]'}")
print(f"   - Mínimo de reseñas por producto: {min_item} (requerido >= 3) {'[OK]' if min_item >= 3 else '[ERROR]'}")

# --- TEST 4: Rangos de Calificaciones ---
min_rating = df_inter['rating'].min()
max_rating = df_inter['rating'].max()
nulos_rating = df_inter['rating'].isna().sum()

print(f"\n4. Rango de Calificaciones:")
print(f"   - Rango detectado: [{min_rating}, {max_rating}] {'[OK]' if min_rating >= 1.0 and max_rating <= 5.0 else '[ERROR]'}")
print(f"   - Calificaciones nulas: {nulos_rating} {'[OK]' if nulos_rating == 0 else '[ERROR]'}")

# --- TEST 5: Distribución de Categorías en el Catálogo ---
print(f"\n5. Desglose de productos por categoría:")
resumen_cat = df_prod['categoria'].value_index = df_prod['product_id'].str[:3].map({
    'bk_': 'Books',
    'af_': 'Fashion',
    'bt_': 'Beauty',
    'vg_': 'Video Games',
    'el_': 'Electronics',
    'kn_': 'Kindle'
}).value_counts()

for cat, count in resumen_cat.items():
    pct = (count / len(df_prod)) * 100
    print(f"   - {cat:<12}: {count:>8,} ({pct:>5.1f}%)")

print("=" * 60)
print("Auditoría finalizada.")

Iniciando auditoría de calidad de datos...
1. Unicidad de productos:
   - IDs duplicados en catálogo: 0 [OK]
   - IDs nulos en catálogo: 0 [OK]

2. Integridad Referencial:
   - Productos en interacciones: 220,178
   - Productos en catálogo: 220,178
   - Ítems en interacciones sin ficha en catálogo: 0 [OK]

3. Validación de k-core:
   - Mínimo de reseñas por usuario: 2 (requerido >= 2) [OK]
   - Mínimo de reseñas por producto: 3 (requerido >= 3) [OK]

4. Rango de Calificaciones:
   - Rango detectado: [1.0, 5.0] [OK]
   - Calificaciones nulas: 0 [OK]

5. Desglose de productos por categoría:
   - Books       :   70,840 ( 32.2%)
   - Beauty      :   67,594 ( 30.7%)
   - Electronics :   63,001 ( 28.6%)
   - Video Games :   17,336 (  7.9%)
   - Kindle      :      963 (  0.4%)
   - Fashion     :      444 (  0.2%)
Auditoría finalizada.
